# RX Strategist OCR Demo

Convert a prescription image to text with Gemini Vision OCR, extract a structured prescription, then verify it with the deterministic engine.

Add `GEMINI_API_KEY` to Colab Secrets before running.

In [ ]:
!git clone https://github.com/YOUR_USERNAME/rx-strategist-mvp.git
%pip install -r rx-strategist-mvp/requirements.txt
import sys
sys.path.insert(0, "rx-strategist-mvp/src")

In [ ]:
from pathlib import Path

from google.colab import userdata
from rx_strategist.extraction.gemini_extractor import GeminiPrescriptionExtractor
from rx_strategist.ocr.gemini_ocr import GeminiPrescriptionOCR
from rx_strategist.verification.verifier import verify_prescription

api_key = userdata.get("GEMINI_API_KEY")
ocr = GeminiPrescriptionOCR(api_key=api_key)
extractor = GeminiPrescriptionExtractor(api_key=api_key)

SAMPLE_CANDIDATES = [
    Path("rx-strategist-mvp/data/prescriptions/sample_prescription.png"),
    Path("data/prescriptions/sample_prescription.png"),
]
SAMPLE_IMAGE = next(path for path in SAMPLE_CANDIDATES if path.is_file())
SAMPLE_IMAGE

In [ ]:
raw_prescription = ocr.ocr_image(SAMPLE_IMAGE)
raw_prescription

In [ ]:
# Optional: upload your own prescription photo instead of the sample image.
from google.colab import files

uploaded = files.upload()
if uploaded:
    filename, image_bytes = next(iter(uploaded.items()))
    suffix = Path(filename).suffix.lower()
    mime_types = {
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".webp": "image/webp",
    }
    mime_type = mime_types.get(suffix)
    if not mime_type:
        raise ValueError(f"Unsupported image type: {suffix}")
    raw_prescription = ocr.ocr_image(image_bytes, mime_type=mime_type)
    raw_prescription

In [ ]:
structured_prescription = extractor.extract_prescription(raw_prescription)
structured_prescription

In [ ]:
result = verify_prescription(structured_prescription)
result